In [4]:
from pathlib import Path

def cari_akar_repo(mulai: Path = Path.cwd()) -> Path:
    for p in [mulai, *mulai.parents]:
        if (p / "pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Akar repo tidak ditemukan. Buka notebook dari dalam folder repo.")

ROOT = cari_akar_repo()
RAW = ROOT / "data" / "raw"
PROFIL = ROOT / "reports" / "profil"
GAMBAR = ROOT / "reports" / "gambar"
PROFIL.mkdir(parents=True, exist_ok=True)
GAMBAR.mkdir(parents=True, exist_ok=True)
print("Akar repo:", ROOT.name)

Akar repo: capstone-dw-kelompok-11


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# --- Akar repo otomatis (tanpa path lokal) ---
def cari_akar_repo(mulai=None):
    mulai = Path.cwd() if mulai is None else Path(mulai)
    for p in [mulai, *mulai.parents]:
        if (p / "pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Akar repo tidak ditemukan. Buka notebook dari dalam folder repo.")


ROOT = cari_akar_repo()
sys.path.insert(0, str(ROOT))
from src.common.manifest import sha256_raw   # lineage dari starter, bukan ditulis tangan

RAW = ROOT / "data" / "raw"
PROFIL = ROOT / "reports" / "profil"
GAMBAR = ROOT / "reports" / "gambar"
PROFIL.mkdir(parents=True, exist_ok=True)
GAMBAR.mkdir(parents=True, exist_ok=True)

# sama persis dengan daftar_calon_anomali.csv di main (commit c1ecac2); 79 pada komentar lama tidak dapat dihasilkan ulang
JUMLAH_CALON_SEBELUMNYA = 77

# --- 1. Muat data mentah BPS ---
BERKAS_RAW = "bps/2026-10-17/sdg8_tpt.csv"      # relatif terhadap data/raw/
file_path = RAW / BERKAS_RAW
assert file_path.exists(), f"Berkas tidak ada: {BERKAS_RAW}. Jalankan python unduh_data.py --kasus 11"
SHA_RAW = sha256_raw(BERKAS_RAW)

raw0 = pd.read_csv(file_path)
print("Akar repo:", ROOT.name)
print("Kolom tersedia:", raw0.columns.tolist())


def pilih_kolom(kandidat, arti):
    for k in kandidat:
        if k in raw0.columns:
            return k
    raise KeyError(f"Kolom {arti} tidak ditemukan. Kolom tersedia: {raw0.columns.tolist()}")


K_WIL = pilih_kolom(["nama_wilayah", "wilayah", "provinsi", "nama_provinsi"], "wilayah")
K_PER = pilih_kolom(["periode", "tahun"], "periode")
K_NIL = pilih_kolom(["nilai", "value", "data"], "nilai")
RENAME = {K_WIL: "nama_wilayah", K_PER: "periode", K_NIL: "nilai"}
print("Kolom wilayah/periode/nilai:", K_WIL, K_PER, K_NIL)

# kolom konteks bawaan sumber yang ikut dibawa ke daftar calon
K_EXTRA = [c for c in ["wilayah_id", "tahun", "turvar", "turvar_id", "satuan"] if c in raw0.columns]
print("Kolom konteks ikut dibawa:", K_EXTRA)


def muat_raw():
    """Baca ulang berkas mentah dengan nama kolom yang sama seperti df."""
    return pd.read_csv(file_path).rename(columns=RENAME)


df = muat_raw()

# --- 2. Statistik deskriptif ---
print("\n--- Menghitung Statistik Deskriptif ---")
print("Statistik Global Nilai var543:")
print(df["nilai"].describe())

stats_provinsi = df.groupby("nama_wilayah")["nilai"].agg(
    count="count",
    mean="mean",
    std="std",
    min="min",
    q1=lambda x: x.quantile(0.25),
    median="median",
    q3=lambda x: x.quantile(0.75),
    max="max",
).reset_index()
stats_provinsi.to_csv(PROFIL / "statistik_provinsi_var543.csv", index=False)
print("Ringkasan per wilayah disimpan:", (PROFIL / "statistik_provinsi_var543.csv").relative_to(ROOT).as_posix())

# --- 3. Deteksi calon anomali (IQR dan z-score; hanya ditandai, tidak diubah) ---
print("\n--- Mendeteksi Calon Anomali ---")
Q1 = df["nilai"].quantile(0.25)
Q3 = df["nilai"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df["anomali_iqr"] = (df["nilai"] < lower_bound) | (df["nilai"] > upper_bound)

mean_val = df["nilai"].mean()
std_val = df["nilai"].std()
df["z_score"] = (df["nilai"] - mean_val) / std_val
df["anomali_zscore"] = df["z_score"].abs() > 3

df_anomali = df[df["anomali_iqr"] | df["anomali_zscore"]].copy()
df_anomali["alasan"] = np.where(
    df_anomali["nilai"] > upper_bound, "Ekstrem Tinggi (> Q3 + 1.5*IQR)",
    np.where(df_anomali["nilai"] < lower_bound, "Ekstrem Rendah (< Q1 - 1.5*IQR)", "Z-Score > 3"),
)

# jejak ke observasi asal (index df = nomor baris data di CSV mentah, mulai dari 0)
df_anomali["berkas_raw"] = BERKAS_RAW
df_anomali["sha256_raw"] = SHA_RAW
df_anomali["indeks_raw"] = df_anomali.index
df_anomali["baris_csv"] = df_anomali.index + 2     # header = baris 1 (benar jika tidak ada sel multibaris)
df_anomali.insert(0, "id_calon", [f"A{i:03d}" for i in range(1, len(df_anomali) + 1)])

kolom_info = list(dict.fromkeys(
    ["id_calon", "nama_wilayah", *K_EXTRA, "periode", "nilai", "z_score",
     "anomali_iqr", "anomali_zscore", "alasan",
     "berkas_raw", "sha256_raw", "indeks_raw", "baris_csv"]
))
daftar_anomali = df_anomali[kolom_info].reset_index(drop=True)

assert daftar_anomali["nama_wilayah"].notna().all(), "Ada calon tanpa wilayah"
assert len(daftar_anomali) == JUMLAH_CALON_SEBELUMNYA, (
    f"Jumlah calon {len(daftar_anomali)} berbeda dari {JUMLAH_CALON_SEBELUMNYA}; periksa sebelum lanjut")

daftar_anomali.to_csv(PROFIL / "daftar_calon_anomali.csv", index=False)
print(f"Ditemukan {len(daftar_anomali)} calon anomali. Disimpan: reports/profil/daftar_calon_anomali.csv")


# --- 4. Penelusuran tiap calon ke observasi asal ---
def telusuri(id_calon):
    """Verifikasi sidik berkas mentah, lalu tampilkan baris asal calon."""
    c = daftar_anomali.set_index("id_calon").loc[id_calon]
    assert sha256_raw(c["berkas_raw"]) == c["sha256_raw"], "sidik berkas mentah tidak cocok"
    baris = muat_raw().iloc[[int(c["indeks_raw"])]]
    assert np.isclose(float(baris["nilai"].iloc[0]), float(c["nilai"])), "nilai tidak cocok"
    assert baris["nama_wilayah"].iloc[0] == c["nama_wilayah"], "wilayah tidak cocok"
    for k in K_EXTRA:
        assert str(baris[k].iloc[0]) == str(c[k]), f"kolom {k} tidak cocok"
    return baris


terlacak = sum(len(telusuri(i)) == 1 for i in daftar_anomali["id_calon"])
print(f"Terlacak ke berkas mentah: {terlacak}/{len(daftar_anomali)}")
assert terlacak == len(daftar_anomali)

# --- 5. Visualisasi ---
print("\n--- Membuat Grafik Visualisasi ---")
sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="nama_wilayah", y="nilai", hue="nama_wilayah", palette="Set2", legend=False)
plt.xticks(rotation=90, fontsize=8)
plt.title("Boxplot Tingkat Pengangguran Terbuka (var543)")
plt.xlabel("Wilayah / Provinsi")
plt.ylabel("Nilai (%)")
plt.tight_layout()
box_path = GAMBAR / "boxplot_var543.png"
plt.savefig(box_path, dpi=300)
plt.close()
print("Boxplot disimpan:", box_path.relative_to(ROOT).as_posix())

plt.figure(figsize=(14, 6))
sns.lineplot(data=df, x="periode", y="nilai", hue="nama_wilayah", alpha=0.7, legend=False)
plt.title("Deret Waktu Tingkat Pengangguran Terbuka (var543)")
plt.xlabel("Periode")
plt.ylabel("Nilai (%)")
plt.tight_layout()
ts_path = GAMBAR / "timeseries_var543.png"
plt.savefig(ts_path, dpi=300)
plt.close()
print("Grafik deret waktu disimpan:", ts_path.relative_to(ROOT).as_posix())

Akar repo: capstone-dw-kelompok-11
Kolom tersedia: ['var_id', 'wilayah_id', 'wilayah', 'turvar_id', 'turvar', 'tahun_id', 'tahun', 'periode_id', 'periode', 'nilai', 'satuan']
Kolom wilayah/periode/nilai: wilayah periode nilai
Kolom konteks ikut dibawa: ['wilayah_id', 'tahun', 'turvar', 'turvar_id', 'satuan']

--- Menghitung Statistik Deskriptif ---
Statistik Global Nilai var543:
count    2000.000000
mean        5.637525
std         2.925345
min         0.560000
25%         3.590000
50%         5.080000
75%         7.020000
max        18.910000
Name: nilai, dtype: float64
Ringkasan per wilayah disimpan: reports/profil/statistik_provinsi_var543.csv

--- Mendeteksi Calon Anomali ---
Ditemukan 77 calon anomali. Disimpan: reports/profil/daftar_calon_anomali.csv
Terlacak ke berkas mentah: 77/77

--- Membuat Grafik Visualisasi ---
Boxplot disimpan: reports/gambar/boxplot_var543.png
Grafik deret waktu disimpan: reports/gambar/timeseries_var543.png


In [ ]:
import io, subprocess
from collections import Counter

hasil = subprocess.run(
    ["git", "show", "origin/main:reports/profil/daftar_calon_anomali.csv"],
    capture_output=True, text=True, cwd=ROOT)
lama = pd.read_csv(io.StringIO(hasil.stdout))
print("Daftar lama:", len(lama), "baris | kolom:", lama.columns.tolist())
print("Daftar baru:", len(daftar_anomali), "baris")

kunci_lama = Counter(zip(lama["periode"].astype(str), lama["nilai"].round(4), lama["alasan"]))
kunci_baru = Counter(zip(daftar_anomali["periode"].astype(str), daftar_anomali["nilai"].round(4), daftar_anomali["alasan"]))
print("\nAda di lama, tidak di baru:", dict(kunci_lama - kunci_baru))
print("Ada di baru, tidak di lama:", dict(kunci_baru - kunci_lama))

print("\nBatas IQR:", round(lower_bound, 4), round(upper_bound, 4), "| jumlah baris df:", len(df))
print(df["nilai"].isna().sum(), "nilai kosong")

Daftar lama: 77 baris | kolom: ['periode', 'nilai', 'alasan']
Daftar baru: 77 baris

Ada di lama, tidak di baru: {}
Ada di baru, tidak di lama: {}

Batas IQR: -1.555 12.165 | jumlah baris df: 2000
0 nilai kosong


In [6]:
daftar_anomali["nama_wilayah"].str.contains("Indonesia", case=False).sum()

np.int64(0)